# 🧠 NeuroMail Colab Brain
**Run this notebook to use Google's free T4 GPU as your AI brain.**

### Steps:
1. Make sure **Runtime → Change runtime type → T4 GPU** is selected
2. Run all cells (Runtime → Run all)
3. Copy the **ngrok URL** printed at the end
4. Paste it in NeuroMail **Settings → AI Brain → Colab Brain**
5. Click **Test** to verify the connection

> ⚡ Expected speed: **2-5 seconds** per response (vs 5 minutes on CPU)

In [ ]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
!apt-get update && apt-get install -y zstd
!pip install -q fastapi uvicorn pyngrok nest-asyncio httpx
print('✅ Dependencies installed')

In [ ]:
# ============================================================
# CELL 2: Install Ollama
# ============================================================
import subprocess
import os

print('📦 Installing Ollama...')
result = subprocess.run(
    'curl -fsSL https://ollama.com/install.sh | sh',
    shell=True, capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else 'Done')
print('✅ Ollama installed')

In [ ]:
# ============================================================
# CELL 3: Start Ollama server in background
# ============================================================
import subprocess
import time

# Start Ollama server
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)  # Wait for server to start
print('✅ Ollama server started')

In [ ]:
# ============================================================
# CELL 4: Pull the AI model
# ============================================================
# You can change this to any model you want:
# - llama3.2:latest (3B, fast, great quality)
# - qwen2.5:7b (7B, better quality, slower)
# - mistral:7b (7B, excellent reasoning)

MODEL = 'llama3.2:latest'  # Change this if you want a different model

print(f'📥 Pulling {MODEL}... (this may take a few minutes)')
result = subprocess.run(
    ['ollama', 'pull', MODEL],
    capture_output=True, text=True
)
print(result.stdout[-300:] if result.stdout else 'Done')
print(f'✅ Model {MODEL} ready!')

In [ ]:
# ============================================================
# CELL 5: Create the FastAPI bridge server
# ============================================================
import nest_asyncio
import asyncio
import httpx
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, StreamingResponse
import uvicorn
import threading

nest_asyncio.apply()

app = FastAPI(title='NeuroMail Colab Brain')
OLLAMA_URL = 'http://localhost:11434'

@app.get('/health')
async def health():
    return {'status': 'ok', 'model': MODEL, 'gpu': 'T4'}

@app.post('/api/chat')
async def chat(request: Request):
    body = await request.json()
    stream = body.get('stream', False)

    if stream:
        async def generate():
            async with httpx.AsyncClient(timeout=300) as client:
                async with client.stream(
                    'POST',
                    f'{OLLAMA_URL}/api/chat',
                    json=body
                ) as response:
                    async for chunk in response.aiter_bytes():
                        yield chunk
        return StreamingResponse(generate(), media_type='application/x-ndjson')
    else:
        async with httpx.AsyncClient(timeout=300) as client:
            response = await client.post(f'{OLLAMA_URL}/api/chat', json=body)
            return JSONResponse(content=response.json(), status_code=response.status_code)

@app.post('/api/generate')
async def generate_endpoint(request: Request):
    body = await request.json()
    async with httpx.AsyncClient(timeout=300) as client:
        response = await client.post(f'{OLLAMA_URL}/api/generate', json=body)
        return JSONResponse(content=response.json(), status_code=response.status_code)

# Start server in background thread
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print('✅ FastAPI bridge server started on port 8000')

In [ ]:
# ============================================================
# CELL 6: Create ngrok tunnel & print URL
# ============================================================
from pyngrok import ngrok, conf

# ⚠️ IMPORTANT: Get your free ngrok auth token from https://dashboard.ngrok.com/
# Then paste it below:
NGROK_AUTH_TOKEN = ''  # <-- Paste your ngrok token here

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open tunnel
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print('=' * 60)
print('🚀 NEUROMAIL COLAB BRAIN IS LIVE!')
print('=' * 60)
print(f'\n📋 Copy this URL and paste it in NeuroMail Settings:')
print(f'\n   👉  {public_url}  👈')
print(f'\n🤖 Model: {MODEL}')
print(f'⚡ GPU: T4 (Google Colab)')
print('=' * 60)
print('\n⚠️  Keep this notebook running to maintain the connection!')
print('   The session will timeout after ~12 hours of inactivity.')

In [ ]:
# ============================================================
# CELL 7: Keep alive (run this to prevent Colab timeout)
# ============================================================
import time

print('🔄 Keep-alive running... (Ctrl+C to stop)')
print(f'🌐 URL: {public_url}')

try:
    while True:
        time.sleep(60)
        # Ping our own server to keep it alive
        try:
            import httpx
            r = httpx.get('http://localhost:8000/health', timeout=5)
            print(f'💚 Alive - {time.strftime("%H:%M:%S")} | Status: {r.json()["status"]}')
        except Exception as e:
            print(f'⚠️  Ping failed: {e}')
except KeyboardInterrupt:
    print('\n🛑 Stopped. Run this cell again to resume.')